# 01 — Initial Data Analysis

**Project:** Dynamic Rental Price Prediction through Multi-Platform Data Scraping, EDA, Ensemble ML & Explainable AI

**Purpose:** Load a raw rental dataset from `data/raw/`, inspect its structure, and produce a preliminary profile (shape, columns, types, missing values, statistics, and basic price distribution).

> **Note:** This notebook uses **real data only**. No synthetic or fake data is generated. If no CSV file is found in `data/raw/`, a graceful message is shown instead.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)

RAW_DIR = Path("../data/raw")
print(f"Looking for CSV files in: {RAW_DIR.resolve()}")

## 1 — Locate and Load the Dataset

In [ ]:
csv_files = sorted(RAW_DIR.glob("*.csv"))

if not csv_files:
    print(
        "⚠️  No CSV file found in data/raw/.\n"
        "   Place your rental dataset (e.g. rentals.csv) in that folder and re-run this cell."
    )
    df = None
else:
    data_path = csv_files[0]  # load the first CSV found
    print(f"Loading: {data_path.name}")
    df = pd.read_csv(data_path)
    print(f"✅ Loaded {len(df):,} rows × {df.shape[1]} columns")

## 2 — Quick Inspection

If no data was loaded, the cells below will print a placeholder message and skip analysis.

In [ ]:
if df is not None:
    print(f"Shape: {df.shape}")
    print(f"\nColumn names ({len(df.columns)}):")
    for col in df.columns:
        print(f"  • {col}")
else:
    print("⏭️  Skipped — no data loaded.")

In [ ]:
if df is not None:
    print("First 5 rows:")
    display(df.head())

In [ ]:
if df is not None:
    print("Last 5 rows:")
    display(df.tail())

In [ ]:
if df is not None:
    df.info()

## 3 — Data Types & Missing Values

In [ ]:
if df is not None:
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    dtype_info = df.dtypes

    summary = pd.DataFrame({
        "dtype": dtype_info,
        "missing_count": missing,
        "missing_pct": missing_pct,
    })
    display(summary)
else:
    print("⏭️  Skipped — no data loaded.")

## 4 — Descriptive Statistics

In [ ]:
if df is not None:
    print("Numeric columns:")
    display(df.describe().T)
else:
    print("⏭️  Skipped — no data loaded.")

In [ ]:
if df is not None:
    cat_cols = df.select_dtypes(include="object").columns.tolist()
    if cat_cols:
        print(f"Categorical columns ({len(cat_cols)}):")
        display(df[cat_cols].describe())
    else:
        print("No categorical (object) columns found.")

## 5 — Basic Price Analysis

Auto-detect the most likely price/rent column and plot its distribution. If no price column is found, a message is shown.

In [ ]:
def detect_price_column(dataframe: pd.DataFrame) -> str | None:
    """Heuristic to find the rental price column."""
    candidates = ["rent", "price", "rental_price", "monthly_rent", "Rent", "Price"]
    for name in candidates:
        if name in dataframe.columns:
            return name
    # fallback: first numeric column whose name contains 'rent' or 'price'
    for col in dataframe.select_dtypes(include="number").columns:
        if any(kw in col.lower() for kw in ("rent", "price")):
            return col
    return None


price_col = detect_price_column(df) if df is not None else None

if price_col:
    print(f"Detected price column: '{price_col}'")
    print(df[price_col].describe())
elif df is not None:
    print(
        "⚠️  Could not auto-detect a price/rent column.\n"
        "   Set `price_col = 'your_column_name'` manually and re-run."
    )
else:
    print("⏭️  Skipped — no data loaded.")

In [ ]:
if price_col and df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    sns.histplot(df[price_col].dropna(), kde=True, bins=50, ax=axes[0], color="#4C72B0")
    axes[0].set_title(f"Distribution of {price_col}")
    axes[0].set_xlabel(price_col)
    axes[0].set_ylabel("Count")

    # Box plot
    sns.boxplot(x=df[price_col].dropna(), ax=axes[1], color="#55A868")
    axes[1].set_title(f"Box Plot of {price_col}")
    axes[1].set_xlabel(price_col)

    plt.tight_layout()
    plt.show()
else:
    print("⏭️  Price visualisation skipped.")

## 6 — Correlation Heatmap (Numeric Columns)

In [ ]:
if df is not None:
    num_df = df.select_dtypes(include="number")
    if num_df.shape[1] >= 2:
        plt.figure(figsize=(12, 8))
        sns.heatmap(
            num_df.corr(),
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            square=True,
            linewidths=0.5,
        )
        plt.title("Correlation Matrix — Numeric Features")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough numeric columns for a correlation matrix.")
else:
    print("⏭️  Skipped — no data loaded.")

---

## Summary

| Check | Status |
|-------|--------|
| Dataset loaded | ✅ / ⏭️ |
| Shape inspected | ✅ / ⏭️ |
| Missing values reported | ✅ / ⏭️ |
| Descriptive statistics | ✅ / ⏭️ |
| Price distribution plotted | ✅ / ⏭️ |
| Correlation heatmap | ✅ / ⏭️ |

**Next:** Place a real dataset in `data/raw/`, re-run this notebook, then proceed to `02_eda.ipynb` for deeper exploratory analysis.